# Notebook 04 — Flood Event Viewer + Stakeholder Dashboard
## Cagayan River Basin | GloFAS Impact Forecast Pipeline

**Purpose**: Communicate four named historical flood events to non-technical stakeholders.
Shows modelled flood depth, estimated people at risk, and event severity.

**Key differences from NB03 (Validation)**

| | NB03 | NB04 |
|---|---|---|
| Satellite data | GFM observed extents | ❌ Not used |
| Accuracy metrics | IoU, F1, Recall | ❌ Not shown |
| Depth threshold | Slider (0–3 m) | Fixed 0.2 m |
| Events | Auto-detected GFM | 4 named events |
| RP communication | Discharge RP | Population RP (EVT2) |

---
## Design decisions

| ID | Decision |
|---|---|
| **D1** | **Flood day selection**: a day is active if ≥1 gauge has RP ≥ 1 yr. Envelope = pixel-max RP over all active days (exact NB03 logic). Peak = day with the most active gauges; tie-break = highest median RP on that day. Fallback: full window if no active days found (warning issued). Rationale: most hillslope/headwater cells never flood, so a *median* RP criterion would miss real events — NB03 uses the same any-gauge rule implicitly. |
| **D2** | Outputs in `data/processed/event_viewer/{BASIN_ID}/{RUN_TAG}/` — separate from NB03 so notebooks never overwrite each other. |
| **D3** | Depth colour scale: Blues, vmin=0.2 m, vmax=99th-percentile of flooded pixels. No fixed cap — depth can reach 20+ m. |
| **D4** | Severity badge: alert vocabulary aligned with NB5 Risk Profile. Bands: RP<2 Moderate, RP2–5 High, RP≥5 Very High. |
| **D5** | Event RP classification uses the NB5 watershed-level OEP curve (10,000-yr YLT simulation). NB5 must run first. EVT2 parameters are no longer used directly for RP assignment. |
| **D6** | FLOPROS computed but hidden from dashboard (same policy as NB03). |
| **D7** | Dashboard: two-layer map toggle (Flood Depth ↔ Affected Population). No slider. |
| **D8** | ADM3 choropleth uses unified population scale across all events for comparability. |


In [1]:
from __future__ import annotations

import os, re, json, math, base64
from pathlib import Path
from typing import Dict, List, Tuple, Optional
from io import BytesIO

import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from shapely.geometry import box

import rasterio
from rasterio.merge import merge as rio_merge
from rasterio.mask import mask as rio_mask
from rasterio.warp import reproject, Resampling
from rasterio.features import geometry_mask

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from PIL import Image

try:
    from IPython.display import display
except Exception:
    display = print

CLIMADA_AVAILABLE = True
try:
    from climada_petals.hazard.rf_glofas.transform_ops import (
        regrid as petals_regrid,
        flood_depth as petals_flood_depth,
        apply_flopros as petals_apply_flopros,
    )
    from climada_petals.hazard.rf_glofas.setup import download_flopros_database
except Exception as _e:
    CLIMADA_AVAILABLE = False
    print('❌ CLIMADA-Petals not available:', _e)

try:
    from philflood.calibration.evt_pot import discharge_to_return_period_pot
except Exception as _e:
    raise ImportError('discharge_to_return_period_pot not importable') from _e

print('✅ Imports complete.')


✅ Imports complete.


In [2]:
# ── Repo root auto-detection (mirrors NB03 Cell 5) ─────────────────────────
REPO_ROOT_MANUAL = None  # override: Path(r'C:\pipelines\...')

def _find_repo_root(start=None):
    markers = ['pyproject.toml', 'setup.cfg', 'setup.py', '.git', 'src']
    p = (Path(start) if start else Path.cwd()).resolve()
    if p.is_file(): p = p.parent
    for parent in (p, *p.parents):
        if any((parent / m).exists() for m in markers):
            return parent
    raise RuntimeError('Repo root not found. Set REPO_ROOT_MANUAL.')

if REPO_ROOT_MANUAL is None:
    try:
        REPO_ROOT = _find_repo_root()
    except Exception:
        REPO_ROOT = Path(r'C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL')
else:
    REPO_ROOT = Path(REPO_ROOT_MANUAL)
REPO_ROOT = REPO_ROOT.resolve()
print('REPO_ROOT =', REPO_ROOT)

DATA_ROOT      = REPO_ROOT / 'data'
RAW_ROOT       = DATA_ROOT / 'raw'
PROCESSED_ROOT = DATA_ROOT / 'processed'
DATA_INTERIM   = DATA_ROOT / 'interim'

# ── Fixed raw-data paths (inherited from NB03 Cell 5) ────────────────────────
HYBAS_L7_SHP    = Path(r'C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\vectors\hydrobasins\australasia\hybas_au_lev01-12_v1c\hybas_au_lev07_v1c.shp')
ADM3_GEOJSON    = Path(r'C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\vectors\admin\phl_cod_ab\phl_adm3.geojson')
WORLDPOP_RASTER = Path(r'C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\worldpop\PHL\phl_pop_2025_CN_100m_R2025A_v1.tif')
JRC_RAW_ROOT    = Path(r'C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\raw\jrc_flood_maps')

# ── Auto-detect calibration run (mirrors NB03 Cell 8) ────────────────────────
AUTO_DETECT   = True
BASIN_ID_HINT = None
RUN_TAG_HINT  = None

def _find_run_config(processed_root, basin_hint=None, run_hint=None):
    calib_root = processed_root / 'calibration' / 'evt_pot'
    if basin_hint and run_hint:
        p = calib_root / basin_hint / run_hint / 'run_config.json'
        if not p.exists(): raise FileNotFoundError(f'run_config.json not found: {p}')
        return p
    candidates = list(calib_root.glob('*/*/run_config.json'))
    if basin_hint: candidates = [c for c in candidates if c.parent.parent.name == basin_hint]
    if not candidates: raise FileNotFoundError(f'No run_config.json under {calib_root}')
    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]

if AUTO_DETECT:
    _rc_path = _find_run_config(PROCESSED_ROOT, BASIN_ID_HINT, RUN_TAG_HINT)
    _rc      = json.loads(_rc_path.read_text(encoding='utf-8'))
    USE_MUNI_AOI   = bool(_rc.get('use_muni_aoi', _rc.get('selection_mode','basin')=='municipality'))
    BASIN_ID       = _rc.get('basin_id', 'MUNI_SELECTION')
    RUN_TAG        = _rc.get('run_tag')
    _sm            = _rc.get('adm3_ids', _rc.get('selected_municipalities', []))
    SELECTED_MUNIS = _sm if isinstance(_sm, list) else []
    print('✅ Auto-detected:', _rc_path)
else:
    USE_MUNI_AOI   = True
    BASIN_ID       = BASIN_ID_HINT or 'MUNI_SELECTION'
    RUN_TAG        = RUN_TAG_HINT  or 'latest'
    SELECTED_MUNIS = []

# ── Derived paths ──────────────────────────────────────────────────────────
EVT_PARAMS_PARQUET = PROCESSED_ROOT / 'calibration/evt_pot' / BASIN_ID / RUN_TAG / 'results/evt_pot_calibration.parquet'
TIMESERIES_DIR     = PROCESSED_ROOT / 'calibration/evt_pot' / BASIN_ID / RUN_TAG / 'timeseries'
OUTPUT_DIR         = PROCESSED_ROOT / 'impact_catalogue_catmodel'
EVT2_JSON_PATH     = OUTPUT_DIR / 'evt2/evt2_fit_popaffected_op.json'
EVT2_RL_PARQUET    = OUTPUT_DIR / 'evt2/evt2_return_levels_popaffected_op.parquet'
WATERSHED_OEP_CURVE_PATH = PROCESSED_ROOT / 'Riskprofiles' / 'watershed_oep_curve.json'
OUT_DIR            = PROCESSED_ROOT / 'event_viewer' / BASIN_ID / RUN_TAG  # Decision D2
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'BASIN_ID           = {BASIN_ID}')
print(f'RUN_TAG            = {RUN_TAG}')
print(f'OUT_DIR            = {OUT_DIR}')
print(f'EVT_PARAMS_PARQUET = {EVT_PARAMS_PARQUET}')
print(f'EVT2_JSON_PATH     = {EVT2_JSON_PATH}')
print(f'WATERSHED_OEP_CURVE_PATH = {WATERSHED_OEP_CURVE_PATH}')

REPO_ROOT = C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL
✅ Auto-detected: C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\processed\calibration\evt_pot\Cagayan_01\2026-01-19_calib-test\run_config.json
BASIN_ID           = Cagayan_01
RUN_TAG            = 2026-01-19_calib-test
OUT_DIR            = C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\processed\event_viewer\Cagayan_01\2026-01-19_calib-test
EVT_PARAMS_PARQUET = C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\processed\calibration\evt_pot\Cagayan_01\2026-01-19_calib-test\results\evt_pot_calibration.parquet
EVT2_JSON_PATH     = C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\processed\impact_catalogue_catmodel\evt2\evt2_fit_popaffected_op.json
WATERSHED_OEP_CURVE_PATH = C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\processed\Riskprofiles\watershed_oep_curve.json


In [3]:
print('=' * 60)
print('Pre-flight check')
print('=' * 60)
_OPT = {'EVT2 fit JSON'}  # EVT2 is optional (used as fallback context only); OEP curve is required
_checks = [
    ('REPO_ROOT exists',           REPO_ROOT.exists()),
    ('ADM3 GeoJSON',               ADM3_GEOJSON.exists()),
    ('WorldPop raster',            WORLDPOP_RASTER.exists()),
    ('HYBAS L7 shapefile',         HYBAS_L7_SHP.exists()),
    ('EVT params parquet',         EVT_PARAMS_PARQUET.exists()),
    ('TIMESERIES_DIR',             TIMESERIES_DIR.exists()),
    ('JRC raw root',               JRC_RAW_ROOT.exists()),
    ('CLIMADA-Petals',             CLIMADA_AVAILABLE),
    ('EVT2 fit JSON',              EVT2_JSON_PATH.exists()),
    ('NB5 watershed OEP curve',    WATERSHED_OEP_CURVE_PATH.exists()),
    ('EVT2 return levels parquet', EVT2_RL_PARQUET.exists()),
]
_fail = False
for label, ok in _checks:
    icon = '✅' if ok else ('⚠️' if label in _OPT else '❌')
    print(f'  {icon}  {label}')
    if not ok and label not in _OPT: _fail = True

if _fail:
    raise RuntimeError('Pre-flight failed — fix paths and re-run.')

if not EVT2_JSON_PATH.exists():
    print()
    print('  ℹ️  EVT2 fit JSON missing.')
    print('     Run parquet_save_fix.py once to add hist to the JSON.')
    print('     Fallback: RP interpolated from return-levels table.')

print('\n✅ Pre-flight complete.')


Pre-flight check
  ✅  REPO_ROOT exists
  ✅  ADM3 GeoJSON
  ✅  WorldPop raster
  ✅  HYBAS L7 shapefile
  ✅  EVT params parquet
  ✅  TIMESERIES_DIR
  ✅  JRC raw root
  ✅  CLIMADA-Petals
  ✅  EVT2 fit JSON
  ✅  NB5 watershed OEP curve
  ✅  EVT2 return levels parquet

✅ Pre-flight complete.


In [4]:
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
#  USER CONTROLS — edit here before running
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# --- Intermediate file for NB5 Risk Matrix integration ---
# NB5 reads this file to plot named events on the RISK MATRIX EP chart.
# Path must match NB6_EVENTS_JSON in NB5 Cell 2.
EVENTS_FOR_RISKMATRIX_PATH = PROCESSED_ROOT / "event_viewer" / "events_for_risk_matrix.json"

# --- NB5 watershed OEP curve (REQUIRED — run NB5 first) ---
# NB6 uses this curve to classify named event severity by inverting OEP(RP) → RP(pop).
# This replaces the previous EVT2-only approach; the OEP curve is the authoritative
# frequency model output from the 10,000-year YLT simulation.
WATERSHED_OEP_CURVE_PATH = PROCESSED_ROOT / "Riskprofiles" / "watershed_oep_curve.json"

# Named events.  date_start/date_end = meteorological window (user-provided).
# STY Uwan end-year corrected 2020→2025 per user confirmation (Q2).
NAMED_EVENTS = {
    'EV_NEM2025': {
        'label':      'Shearline / NE Monsoon — Dec 2025',
        'date_start': '2025-11-19',
        'date_end':   '2025-11-30',
    },
    'EV_ULYSSES2020': {
        'label':      'TY Ulysses — Nov 2020',
        'date_start': '2020-11-08',
        'date_end':   '2020-11-15',
    },
    'EV_UWAN2025': {
        'label':      'STY Uwan — Nov 2025',
        'date_start': '2025-11-04',
        'date_end':   '2025-11-12',
    },
    'EV_MARCE2024': {
        'label':      'TY Marce — Nov 2024',
        'date_start': '2024-11-02',
        'date_end':   '2024-11-12',
    },
}

DISCHARGE_PAD_DAYS  = 3       # days before event window to include in RP search
DEPTH_THRESHOLD_M   = 0.02    # fixed from NB03 analysis (Q3)  — Decision D3
DEPTH_BAND_SHALLOW  = 0.50    # < SHALLOW → 'Shallow'
DEPTH_BAND_MODERATE = 1.50    # [SHALLOW, MODERATE) → 'Moderate'; ≥ MODERATE → 'Deep'

FORCE_RECOMPUTE          = True   # set True to ignore cached .nc/.tif files
ENABLE_FLOPROS_SCENARIOS = True    # computed but hidden from dashboard (Decision D6)
DASHBOARD_SCENARIO       = 'NoProt'
REGRID_METHOD            = 'bilinear'
JRC_RETURN_PERIODS       = [10, 20, 50, 75, 100, 200, 500]
USE_RECLASSIFIED         = False

BASIN_DISPLAY_NAME = 'Cagayan River Basin'
DASHBOARD_FILENAME = OUT_DIR / 'event_viewer_dashboard.html'

# ═══════════════════════════════════════════════════════════════════════════════
# LOAD BASIN CONFIGURATION TO CONSTRAIN AOI (memory optimization)
# ═══════════════════════════════════════════════════════════════════════════════
# Load the HydroBASINS ID and level from basin config to ensure small, focused AOI.
# This is critical for memory efficiency with large JRC tiles.

HYBAS_ID_FIELD  = 'HYBAS_ID'  # Column name in HydroBASINS shapefile
HYBAS_ID_VALUES = None  # Will be populated from basin config

import yaml
basin_config_path = REPO_ROOT / 'ops' / 'configs' / 'basins' / f'{BASIN_ID}.yaml'

if basin_config_path.exists():
    with open(basin_config_path, 'r') as f:
        basin_config = yaml.safe_load(f)

    if 'hydrobasins_id' in basin_config:
        HYBAS_ID_VALUES = [basin_config['hydrobasins_id']]
        _hybas_lvl = basin_config.get('hydrobasins_level', 7)
        _hybas_dir = HYBAS_L7_SHP.parent
        HYBAS_L7_SHP = _hybas_dir / f'hybas_au_lev{_hybas_lvl:02d}_v1c.shp'
        print(f'✅ Loaded basin config: {BASIN_ID}')
        print(f'   HydroBASINS level  : {_hybas_lvl} → {HYBAS_L7_SHP.name}')
        print(f'   HydroBASINS ID     : {HYBAS_ID_VALUES[0]}')
        print(f'   This constrains AOI to the specific basin (memory-efficient)')
    else:
        print(f'⚠️  Basin config found but no hydrobasins_id — will use GFM bounds')
else:
    print(f'⚠️  No basin config at {basin_config_path} — will use GFM bounds fallback')

print(f'\nNamed events:')
for ev_id, cfg in NAMED_EVENTS.items():
    print(f'  {ev_id}: {cfg["label"]}')
    print(f'    {cfg["date_start"]} → {cfg["date_end"]}')
print(f'\nDepth threshold : {DEPTH_THRESHOLD_M} m (fixed, from NB03 Q3)')
print(f'Discharge pad   : {DISCHARGE_PAD_DAYS} days')
print(f'Dashboard output: {DASHBOARD_FILENAME}')


✅ Loaded basin config: Cagayan_01
   HydroBASINS level  : 6 → hybas_au_lev06_v1c.shp
   HydroBASINS ID     : 5060030230
   This constrains AOI to the specific basin (memory-efficient)

Named events:
  EV_NEM2025: Shearline / NE Monsoon — Dec 2025
    2025-11-19 → 2025-11-30
  EV_ULYSSES2020: TY Ulysses — Nov 2020
    2020-11-08 → 2020-11-15
  EV_UWAN2025: STY Uwan — Nov 2025
    2025-11-04 → 2025-11-12
  EV_MARCE2024: TY Marce — Nov 2024
    2024-11-02 → 2024-11-12

Depth threshold : 0.02 m (fixed, from NB03 Q3)
Discharge pad   : 3 days
Dashboard output: C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\processed\event_viewer\Cagayan_01\2026-01-19_calib-test\event_viewer_dashboard.html


In [5]:
# Verbatim from NB03 Cell 12 — no logic changes.
def load_adm3(path: Path) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    gdf = gdf.set_crs('EPSG:4326') if gdf.crs is None else gdf.to_crs('EPSG:4326')
    assert 'adm3_name' in gdf.columns and 'adm3_id' in gdf.columns
    return gdf

adm3_gdf = load_adm3(ADM3_GEOJSON)
print('ADM3 loaded:', len(adm3_gdf))

def _gfm_bounds_union(root: Path):
    # Scan any available GFM tifs for basin bbox fallback.
    tifs = list(root.rglob('*.tif')) if root.exists() else []
    bounds = None
    for p in tifs:
        try:
            with rasterio.open(p) as src: b = src.bounds
        except Exception: continue
        if bounds is None: bounds = [b.left, b.bottom, b.right, b.top]
        else:
            bounds[0]=min(bounds[0],b.left);  bounds[1]=min(bounds[1],b.bottom)
            bounds[2]=max(bounds[2],b.right); bounds[3]=max(bounds[3],b.top)
    return tuple(bounds) if bounds else None

_GFM_ROOT = DATA_INTERIM / 'validation/GFM/Cagayan'
GFM_BOUNDS = _gfm_bounds_union(_GFM_ROOT)
print('GFM bounds:', GFM_BOUNDS or 'not found — will fall back')

# NOTE: HYBAS_ID_FIELD and HYBAS_ID_VALUES are set in Cell 4 (User Controls)
# after loading the basin configuration. This ensures a focused, memory-efficient AOI.
BASIN_BBOX_BUFFER_DEG = 0.10

def build_aoi_boundary(use_muni, selected_munis, adm3_gdf):
    if use_muni:
        if not selected_munis: raise ValueError('Municipality mode requires SELECTED_MUNIS.')
        sel = adm3_gdf[adm3_gdf['adm3_id'].isin(selected_munis)].copy()
        if sel.empty: sel = adm3_gdf[adm3_gdf['adm3_name'].isin(selected_munis)].copy()
        if sel.empty: raise ValueError(f'No ADM3 matched SELECTED_MUNIS: {selected_munis}')
        print(f'✅ Municipality mode: {len(sel)} selected')
        return sel.unary_union, 'municipality'
    hy = gpd.read_file(HYBAS_L7_SHP)
    hy = hy.to_crs('EPSG:4326') if hy.crs else hy.set_crs('EPSG:4326')
    if HYBAS_ID_FIELD and HYBAS_ID_VALUES is not None:
        hy_sel = hy[hy[HYBAS_ID_FIELD].isin(HYBAS_ID_VALUES)].copy()
        if hy_sel.empty: raise ValueError('HYBAS_ID_VALUES matched nothing.')
        print(f'✅ Basin AOI (HydroBASINS ID): {len(hy_sel)} polygons from {HYBAS_L7_SHP.name}')
        return hy_sel.unary_union, 'basin'
    if GFM_BOUNDS:
        w, s, e, n = GFM_BOUNDS
        bbox   = box(w-BASIN_BBOX_BUFFER_DEG, s-BASIN_BBOX_BUFFER_DEG,
                     e+BASIN_BBOX_BUFFER_DEG, n+BASIN_BBOX_BUFFER_DEG)
        hy_sel = hy[hy.intersects(bbox)].copy()
    else:
        hy_sel = hy[hy.intersects(adm3_gdf.unary_union)].copy()
    if hy_sel.empty: raise ValueError(f'No HydroBASINS polygons found for AOI in {HYBAS_L7_SHP.name}.')
    print(f'✅ Basin AOI: {len(hy_sel)} HydroBASINS polygons from {HYBAS_L7_SHP.name}')
    return hy_sel.unary_union, 'basin'

AOI_BOUNDARY, MODE = build_aoi_boundary(USE_MUNI_AOI, SELECTED_MUNIS, adm3_gdf)
print('MODE =', MODE)


ADM3 loaded: 1647
GFM bounds: (-68.390869, -17.802582, 122.85039896498888, 20.241939999999975)
✅ Basin AOI (HydroBASINS ID): 1 polygons from hybas_au_lev06_v1c.shp
MODE = basin


In [6]:
# Verbatim from NB03 Cell 16 — no logic changes.
if not EVT_PARAMS_PARQUET.exists():
    raise FileNotFoundError(f'EVT params not found: {EVT_PARAMS_PARQUET}')

evt_params = pd.read_parquet(EVT_PARAMS_PARQUET)
req_cols = {'virtual_gauge_id','threshold_m3s','lambda_events_per_year','gpd_xi','gpd_sigma'}
missing  = req_cols - set(evt_params.columns)
if missing: raise ValueError(f'EVT params missing columns: {missing}')
evt_params = evt_params.copy()
evt_params['virtual_gauge_id'] = evt_params['virtual_gauge_id'].astype(str)
print('EVT gauges:', len(evt_params))

def _parse_lat_lon(gid: str) -> Tuple[float,float]:
    ml = re.search(r'lat_(-?\d+(?:\.\d+)?)', gid)
    mn = re.search(r'lon_(-?\d+(?:\.\d+)?)', gid)
    if not (ml and mn): raise ValueError(f'Cannot parse lat/lon from: {gid}')
    return float(ml.group(1)), float(mn.group(1))

evt_params['lat'] = evt_params['virtual_gauge_id'].apply(lambda s: _parse_lat_lon(s)[0])
evt_params['lon'] = evt_params['virtual_gauge_id'].apply(lambda s: _parse_lat_lon(s)[1])

def _pick_latlon_cols(df, label):
    for lc, nc in [('lat','lon'),('latitude','longitude'),('cell_lat','cell_lon'),('y','x')]:
        if lc in df.columns and nc in df.columns: return lc, nc
    raise ValueError(f'{label} missing lat/lon columns: {list(df.columns)}')

def _grid_spacing(values):
    v = np.sort(values.unique()); d = np.diff(v); d = d[d>0]
    return float(np.median(d)) if len(d) else np.nan

def _build_cell_polygons(df, lc, nc):
    dx = _grid_spacing(df[nc]); dy = _grid_spacing(df[lc])
    if not (np.isfinite(dx) and np.isfinite(dy)):
        raise ValueError('Cannot infer grid resolution')
    geoms = [box(lon-dx/2, lat-dy/2, lon+dx/2, lat+dy/2)
             for lat, lon in zip(df[lc], df[nc])]
    return gpd.GeoSeries(geoms, crs='EPSG:4326'), dx, dy

mapping_dir    = EVT_PARAMS_PARQUET.parent.parent / 'mapping'
l12_vg_path    = mapping_dir / 'l12_virtual_gauge.parquet'
l12_cells_path = mapping_dir / 'l12_cell_coordinates.parquet'
use_aoi_point_filter = False

if l12_vg_path.exists() and l12_cells_path.exists():
    l12_vg    = pd.read_parquet(l12_vg_path)
    l12_cells = pd.read_parquet(l12_cells_path)
    lc, nc    = _pick_latlon_cols(l12_cells, 'l12_cell_coordinates')
    l12_cells = l12_cells[['l12_id', lc, nc]].copy()
    l12_cells['l12_id'] = l12_cells['l12_id'].astype(str)
    l12_polys, _, _ = _build_cell_polygons(l12_cells, lc, nc)
    l12_gdf   = gpd.GeoDataFrame(l12_cells, geometry=l12_polys, crs='EPSG:4326')
    l12_sel   = l12_gdf[l12_gdf.intersects(AOI_BOUNDARY)].copy()
    l12_ids   = set(l12_sel['l12_id'])
    if not l12_ids: raise ValueError('No L12 cells intersect the AOI boundary.')
    l12_vg['l12_id'] = l12_vg['l12_id'].astype(str)
    allowed = set(l12_vg[l12_vg['l12_id'].isin(l12_ids)]['virtual_gauge_id'].astype(str))
    before  = len(evt_params)
    evt_params = evt_params[evt_params['virtual_gauge_id'].isin(allowed)].copy()
    print(f'EVT gauges after L12→AOI filter: {len(evt_params)} / {before} (L12s: {len(l12_ids)})')
elif l12_vg_path.exists():
    l12_vg = pd.read_parquet(l12_vg_path)
    allowed = set(l12_vg['virtual_gauge_id'].astype(str))
    before  = len(evt_params)
    evt_params = evt_params[evt_params['virtual_gauge_id'].isin(allowed)].copy()
    print(f'EVT gauges after L12 mapping filter: {len(evt_params)} / {before}')
else:
    print('⚠️ L12 mapping not found — using AOI point filter.')
    use_aoi_point_filter = True

evt_points = gpd.GeoDataFrame(
    evt_params.copy(),
    geometry=gpd.points_from_xy(evt_params['lon'], evt_params['lat']),
    crs='EPSG:4326')
if use_aoi_point_filter:
    evt_points = evt_points[evt_points.intersects(AOI_BOUNDARY)].copy()
    if len(evt_points) == 0:
        raise ValueError('No EVT virtual gauges intersect the AOI boundary.')
evt_params = evt_points.drop(columns='geometry')

if evt_params['virtual_gauge_id'].str.startswith('VG__').any():
    n = evt_params['virtual_gauge_id'].str.startswith('VG__').sum()
    print(f'⚠️ Filtering {n} VG__ pour-point gauges — keeping CELL__ only')
    evt_params = evt_params[evt_params['virtual_gauge_id'].str.startswith('CELL__')].copy()

if not TIMESERIES_DIR.exists():
    raise FileNotFoundError(f'TIMESERIES_DIR not found: {TIMESERIES_DIR}')

def _ts_path(gid):
    p = TIMESERIES_DIR / f'{gid}.parquet'
    if p.exists(): return p
    m = list(TIMESERIES_DIR.glob(f'{gid}*.parquet'))
    return m[0] if m else None

def load_discharge_series(gid):
    p = _ts_path(gid)
    if p is None: return None
    try:
        df = pd.read_parquet(p)
        if isinstance(df, pd.Series):
            s = df
        elif 'discharge_m3s' in df.columns:
            dc = next((c for c in ['date','time'] if c in df.columns), None)
            idx = pd.to_datetime(df[dc]) if dc else pd.to_datetime(df.index)
            s = pd.Series(df['discharge_m3s'].values, index=idx)
        else:
            s = df.iloc[:, 0]
        if not isinstance(s.index, pd.DatetimeIndex): s.index = pd.to_datetime(s.index)
        s.index = s.index.normalize()
        if s.index.has_duplicates: s = s.groupby(level=0).max()
        return s.sort_index()
    except Exception as exc:
        print(f'⚠️ Error reading timeseries {gid}: {exc}')
        return None

_ts_cache: Dict[str, Optional[pd.Series]] = {}
_missing_gauges: set = set()

def _get_ts(gid):
    if gid not in _ts_cache:
        _ts_cache[gid] = load_discharge_series(gid)
        if _ts_cache[gid] is None and gid not in _missing_gauges:
            _missing_gauges.add(gid)
    return _ts_cache[gid]

def _build_rp_grid(df_vals, fill=np.nan):
    lats = np.sort(df_vals['lat'].unique()); lons = np.sort(df_vals['lon'].unique())
    grid = np.full((len(lats), len(lons)), fill, dtype='float32')
    li = {v:i for i,v in enumerate(lats)}; lo = {v:i for i,v in enumerate(lons)}
    for _, r in df_vals.iterrows(): grid[li[r['lat']], lo[r['lon']]] = r['rp']
    return xr.DataArray(grid, coords={'latitude': lats, 'longitude': lons},
                        dims=('latitude','longitude'), name='return_period')

def _rp_from_q(q, row):
    return float(discharge_to_return_period_pot(
        q=q, u=float(row['threshold_m3s']),
        xi=float(row['gpd_xi']), sigma=float(row['gpd_sigma']),
        lambda_u=float(row['lambda_events_per_year'])))

print(f'\n✅ Gauge loading complete: {len(evt_params)} gauges in AOI')


EVT gauges: 1024
EVT gauges after L12→AOI filter: 1024 / 1024 (L12s: 215)

✅ Gauge loading complete: 1024 gauges in AOI


In [7]:
# ── Flood day selection — Decision D1 ────────────────────────────────────────
#
# Rule: a day is a FLOOD DAY if >= 1 gauge has RP >= 1 yr.
# This mirrors NB03's episode_df construction where GFM flood pixels are
# the analogous 'any pixel flooded' criterion.
#
# Why not 'median RP >= 1 yr':
#   Most virtual gauge cells sit on hillslopes or headwaters that never reach
#   bankfull.  Their RP is always << 1 yr, so the basin-wide MEDIAN RP stays
#   below 1 yr even during a major event.  Using ANY gauge exceeding the
#   threshold avoids this systematic downward bias.
#
# Envelope: pixel-wise max RP across ALL flood days (identical to NB03).
# Peak day: flood day with the most active gauges.
#           Tie-break: highest median RP among active gauges.
# Fallback: if no flood days found, use full window + print warning.

event_results = {}

for ev_id, ev_cfg in NAMED_EVENTS.items():
    label     = ev_cfg['label']
    d_start   = pd.to_datetime(ev_cfg['date_start'])
    d_end     = pd.to_datetime(ev_cfg['date_end'])
    win_start = d_start - pd.Timedelta(days=DISCHARGE_PAD_DAYS)

    print(f'\n{"="*62}')
    print(f'📅  {ev_id}  |  {label}')
    print(f'    Search window: {win_start.date()} → {d_end.date()}')

    days = pd.date_range(win_start, d_end, freq='D')
    daily_rows = []

    for day in days:
        n_active = 0; n_valid = 0; rp_active = []
        for _, row in evt_params.iterrows():
            s = _get_ts(row['virtual_gauge_id'])
            if s is None: continue
            n_valid += 1
            if day not in s.index: continue
            q = float(s.loc[day])
            if not np.isfinite(q): continue
            rp = _rp_from_q(q, row)
            if rp >= 1.0:
                n_active += 1; rp_active.append(rp)
        daily_rows.append({
            'day':        day,
            'n_valid':    n_valid,
            'n_active':   n_active,
            'pct_active': 100.0 * n_active / n_valid if n_valid > 0 else 0.0,
            'median_rp':  float(np.median(rp_active)) if rp_active else 0.0,
            'max_rp':     float(np.max(rp_active))    if rp_active else 0.0,
        })

    stats_df  = pd.DataFrame(daily_rows)
    flood_df  = stats_df[stats_df['n_active'] > 0].copy()

    if flood_df.empty:
        print('  ⚠️  No active-gauge days — using full window as fallback.')
        flood_days = stats_df['day'].tolist()
        peak_row   = stats_df.sort_values(['max_rp','median_rp'], ascending=False).iloc[0]
    else:
        flood_days = flood_df['day'].tolist()
        peak_row   = flood_df.sort_values(['n_active','median_rp'], ascending=False).iloc[0]

    peak_day = peak_row['day']
    fr = f'  ({flood_days[0].date()} → {flood_days[-1].date()})' if flood_days else ''
    print(f'  Flood days : {len(flood_days)}{fr}')
    print(f'  Peak day   : {peak_day.date()}  |  '
          f'{int(peak_row["n_active"])} active gauges  |  '
          f'median RP = {peak_row["median_rp"]:.1f} yr')

    if not flood_df.empty:
        display(flood_df[['day','n_active','pct_active','median_rp','max_rp']]
                .assign(day=lambda d: d['day'].dt.date)
                .set_index('day').round(1))

    event_results[ev_id] = {
        'label':       label,
        'date_start':  d_start,
        'date_end':    d_end,
        'flood_days':  flood_days,
        'peak_day':    peak_day,
        'daily_stats': stats_df,
    }

if _missing_gauges:
    print(f'\n⚠️  {len(_missing_gauges)} gauges had no timeseries files.')
print('\n✅ Flood day selection complete.')



📅  EV_NEM2025  |  Shearline / NE Monsoon — Dec 2025
    Search window: 2025-11-16 → 2025-11-30


  Flood days : 6  (2025-11-25 → 2025-11-30)
  Peak day   : 2025-11-26  |  49 active gauges  |  median RP = 1.1 yr


,n_active,pct_active,median_rp,max_rp
day,,,,
2025-11-25,4,0.4,1.2,1.2
2025-11-26,49,4.8,1.1,1.5
2025-11-27,14,1.4,1.3,1.5
2025-11-28,13,1.3,1.4,1.6
2025-11-29,48,4.7,1.3,2.4
2025-11-30,47,4.6,1.2,2.8



📅  EV_ULYSSES2020  |  TY Ulysses — Nov 2020
    Search window: 2020-11-05 → 2020-11-15
  Flood days : 7  (2020-11-05 → 2020-11-15)
  Peak day   : 2020-11-12  |  924 active gauges  |  median RP = 4.4 yr


,n_active,pct_active,median_rp,max_rp
day,,,,
2020-11-05,2,0.2,1.2,1.2
2020-11-09,1,0.1,1.1,1.1
2020-11-11,305,29.8,1.5,4.4
2020-11-12,924,90.2,4.4,15.6
2020-11-13,693,67.7,2.8,18.5
2020-11-14,255,24.9,1.6,11.3
2020-11-15,42,4.1,1.1,7.3



📅  EV_UWAN2025  |  STY Uwan — Nov 2025
    Search window: 2025-11-01 → 2025-11-12
  Flood days : 4  (2025-11-09 → 2025-11-12)
  Peak day   : 2025-11-10  |  751 active gauges  |  median RP = 2.0 yr


,n_active,pct_active,median_rp,max_rp
day,,,,
2025-11-09,124,12.1,1.2,2.5
2025-11-10,751,73.3,2.0,5.5
2025-11-11,210,20.5,1.4,2.9
2025-11-12,9,0.9,1.1,1.1



📅  EV_MARCE2024  |  TY Marce — Nov 2024
    Search window: 2024-10-30 → 2024-11-12
  Flood days : 11  (2024-10-30 → 2024-11-12)
  Peak day   : 2024-11-12  |  535 active gauges  |  median RP = 1.3 yr


,n_active,pct_active,median_rp,max_rp
day,,,,
2024-10-30,72,7.0,1.9,4.2
2024-10-31,71,6.9,1.6,2.8
2024-11-01,7,0.7,1.4,1.7
2024-11-02,3,0.3,1.1,1.3
2024-11-06,2,0.2,1.2,1.2
2024-11-07,59,5.8,2.0,4.6
2024-11-08,68,6.6,2.1,4.4
2024-11-09,26,2.5,1.2,2.7
2024-11-10,9,0.9,1.6,1.9



✅ Flood day selection complete.


In [8]:
# ── RP maps (envelope + peakday) — mirrors NB03 compute_episode_rp_maps ────
RP_OUT_DIR = OUT_DIR / 'model/return_period'
RP_OUT_DIR.mkdir(parents=True, exist_ok=True)

def _compute_event_rp_maps(ev_id, ev):
    win_start = ev['date_start'] - pd.Timedelta(days=DISCHARGE_PAD_DAYS)
    win_end   = ev['date_end']
    out_env   = RP_OUT_DIR / f'rp__{ev_id}__envelope.nc'
    out_peak  = RP_OUT_DIR / f'rp__{ev_id}__peakday.nc'

    if not FORCE_RECOMPUTE and out_env.exists() and out_peak.exists():
        print(f'  ✅ {ev_id}: using cached RP maps')
        return {'envelope': out_env, 'peakday': out_peak}
    if FORCE_RECOMPUTE:
        out_env.unlink(missing_ok=True); out_peak.unlink(missing_ok=True)

    days = pd.date_range(win_start, win_end, freq='D')
    rp_maps = []; g_data = 0

    for i, day in enumerate(days):
        vals = []
        for _, row in evt_params.iterrows():
            s = _get_ts(row['virtual_gauge_id'])
            if i == 0 and s is not None: g_data += 1
            if s is None or day not in s.index: q = np.nan
            else: q = float(s.loc[day])
            rp = _rp_from_q(q, row) if np.isfinite(q) else np.nan
            vals.append({'lat': row['lat'], 'lon': row['lon'], 'rp': rp})

        df_day = pd.DataFrame(vals)
        df_day.loc[df_day['rp'] < 1.0, 'rp'] = np.nan  # mirrors NB03
        da_day = _build_rp_grid(df_day).expand_dims({'time': [day]})
        rp_maps.append(da_day)

    pct = 100 * g_data / len(evt_params) if len(evt_params) else 0
    print(f'  📊 {ev_id}: {g_data}/{len(evt_params)} gauges ({pct:.0f}%) have timeseries')
    if g_data == 0:
        print(f'  ⚠️  No timeseries data — RP maps will be all-NaN.')

    rp_stack = xr.concat(rp_maps, dim='time')
    rp_env   = rp_stack.max(dim='time', skipna=True)  # envelope = pixel-wise max

    pk = pd.Timestamp(ev['peak_day'])
    if pk in rp_stack.time.values:
        rp_peak = rp_stack.sel(time=pk).drop_vars('time')
    else:
        print(f'  ⚠️  Peak day {pk.date()} not in daily stack — using envelope as peakday.')
        rp_peak = rp_env

    rp_env.to_dataset(name='return_period').to_netcdf(out_env)
    rp_peak.to_dataset(name='return_period').to_netcdf(out_peak)
    return {'envelope': out_env, 'peakday': out_peak}

print(f'🎯 Computing RP maps for {len(event_results)} events …')
rp_products = {}
for ev_id, ev in event_results.items():
    print(f'\n📊 {ev_id}')
    rp_products[ev_id] = _compute_event_rp_maps(ev_id, ev)

events_df = pd.DataFrame([{
    'event_id': ev_id,
    'label':    ev['label'],
    'start':    ev['flood_days'][0] if ev['flood_days'] else ev['date_start'],
    'end':      ev['flood_days'][-1] if ev['flood_days'] else ev['date_end'],
    'peak_day': ev['peak_day'],
    'envelope': str(rp_products[ev_id]['envelope']),
    'peakday':  str(rp_products[ev_id]['peakday']),
} for ev_id, ev in event_results.items()])

print('\n✅ RP maps ready:')
display(events_df[['event_id','label','start','end','peak_day']])


🎯 Computing RP maps for 4 events …

📊 EV_NEM2025
  📊 EV_NEM2025: 1024/1024 gauges (100%) have timeseries

📊 EV_ULYSSES2020
  📊 EV_ULYSSES2020: 1024/1024 gauges (100%) have timeseries

📊 EV_UWAN2025
  📊 EV_UWAN2025: 1024/1024 gauges (100%) have timeseries

📊 EV_MARCE2024
  📊 EV_MARCE2024: 1024/1024 gauges (100%) have timeseries

✅ RP maps ready:


,event_id,label,start,end,peak_day
0,EV_NEM2025,Shearline / NE Monsoon — Dec 2025,2025-11-25,2025-11-30,2025-11-26
1,EV_ULYSSES2020,TY Ulysses — Nov 2020,2020-11-05,2020-11-15,2020-11-12
2,EV_UWAN2025,STY Uwan — Nov 2025,2025-11-09,2025-11-12,2025-11-10
3,EV_MARCE2024,TY Marce — Nov 2024,2024-10-30,2024-11-12,2024-11-12


In [9]:
import requests
from rasterio.merge import merge as rio_merge

# ═══════════════════════════════════════════════════════════════════════════════
# JRC FLOOD HAZARD MAPS - Download & Mosaic
# ═══════════════════════════════════════════════════════════════════════════════
# Downloads JRC return period tiles and merges them into continuous flood maps.
# Matches the proven approach from NB03 Cell 21.
# ═══════════════════════════════════════════════════════════════════════════════

JRC_BASE_URL = 'https://jeodpp.jrc.ec.europa.eu/ftp/jrc-opendata/CEMS-GLOFAS/flood_hazard/'
JRC_RAW_ROOT.mkdir(parents=True, exist_ok=True)

# Download & cache tile index
tile_index_path = JRC_RAW_ROOT / 'tile_extents.geojson'
if not tile_index_path.exists():
    print('Downloading tile_extents.geojson …')
    r = requests.get(JRC_BASE_URL + 'tile_extents.geojson', timeout=120)
    r.raise_for_status()
    tile_index_path.write_bytes(r.content)

# Select tiles intersecting AOI
tiles_gdf = gpd.read_file(tile_index_path).to_crs('EPSG:4326')
tiles_sel = tiles_gdf[tiles_gdf.intersects(AOI_BOUNDARY)].copy()
tiles_sel['tile_code'] = 'ID' + tiles_sel['id'].astype(str) + '_' + tiles_sel['name'].astype(str)
tile_codes = sorted(tiles_sel['tile_code'].unique().tolist())
print(f'Selected JRC tiles: {len(tile_codes)} | example: {tile_codes[:3]}')

# CHECK: Is AOI too large?
aoi_bounds = AOI_BOUNDARY.bounds
aoi_width = aoi_bounds[2] - aoi_bounds[0]
aoi_height = aoi_bounds[3] - aoi_bounds[1]
pixel_size = 0.000833  # degrees (approximately)
est_h = int(aoi_height / pixel_size)
est_w = int(aoi_width / pixel_size)
est_gb = (est_h * est_w * 4) / 1e9  # float32 = 4 bytes

print(f'\n⚠️  WARNING: AOI is large')
print(f'   Bounds: {aoi_bounds}')
print(f'   Size: {aoi_width:.2f}° × {aoi_height:.2f}°')
print(f'   Est. output: {est_h}×{est_w} (~{est_gb:.1f} GB)')

if est_gb > 12:
    print(f'\n🛑 INSUFFICIENT MEMORY for current AOI!')
    print(f'   System needs ~{est_gb:.0f}GB RAM, but typical systems have 8-16GB')
    print(f'\n💡 SOLUTIONS:')
    print(f'   1. Reload earlier cells with smaller AOI')
    print(f'   2.Or use a smaller region (just focus basin)')
    print(f'   3. Or subsample JRC tiles to lower resolution')
    print(f'\n   For now, attempting with available memory (may fail)...')

def _download_jrc_tile(rp, tile_code, rp_dir, suffix):
    """Download a single JRC tile if not already cached."""
    fname = f'{tile_code}_RP{rp}{suffix}'
    dst = rp_dir / fname
    
    if dst.exists() and dst.stat().st_size > 10_000:
        return dst
    
    url = f'{JRC_BASE_URL}RP{rp}/{fname}'
    try:
        r = requests.get(url, stream=True, timeout=300)
        r.raise_for_status()
        dst.parent.mkdir(parents=True, exist_ok=True)
        with open(dst, 'wb') as f:
            for chunk in r.iter_content(1024*1024):
                if chunk:
                    f.write(chunk)
        return dst
    except Exception as e:
        if dst.exists():
            dst.unlink()
        return None

# Build & cache flood maps
flood_maps_nc = OUT_DIR / 'jrc/flood-maps_intermediate.nc'
flood_maps_nc.parent.mkdir(parents=True, exist_ok=True)

if flood_maps_nc.exists():
    flood_maps = xr.open_dataarray(flood_maps_nc)
    print(f'\n✅ Using cached flood_maps: {flood_maps.shape}')
    print(f'   Return periods: {list(flood_maps.return_period.values)}')
else:
    suffix = '_depth_reclass.tif' if USE_RECLASSIFIED else '_depth.tif'
    flood_maps_list = []
    
    for rp in JRC_RETURN_PERIODS:
        print(f'\n📥 RP{rp}: Downloading & merging {len(tile_codes)} tiles...')
        rp_dir = JRC_RAW_ROOT / f'RP{rp}'
        rp_dir.mkdir(parents=True, exist_ok=True)
        
        # Download tiles
        tile_paths = []
        for tc in tile_codes:
            p = _download_jrc_tile(rp, tc, rp_dir, suffix)
            if p:
                tile_paths.append(p)
        
        if not tile_paths:
            print(f'⚠️  RP{rp}: No tiles downloaded')
            continue
        
        # Merge using rasterio (proven approach)
        try:
            srcs = [rasterio.open(str(p)) for p in tile_paths]
            mosaic, out_t = rio_merge(srcs)
            for s in srcs: s.close()
            
            arr = mosaic[0].astype('float32')
            h, w = arr.shape
            
            # Build coordinates
            lons = out_t.c + out_t.a * (np.arange(w) + 0.5)
            lats = out_t.f + out_t.e * (np.arange(h) + 0.5)
            
            da = xr.DataArray(
                arr,
                coords={'latitude': lats, 'longitude': lons},
                dims=('latitude', 'longitude'),
                name='depth'
            ).expand_dims({'return_period': [rp]})
            
            flood_maps_list.append(da)
            print(f'   ✅ Merged: {h}×{w}')
            
        except MemoryError:
            print(f'❌ RP{rp}: Insufficient system RAM (tried to allocate ~12.9GB)')
            print(f'   Current AOI requires too much memory')
            for p in tile_paths:
                if p.exists(): p.unlink()
            continue
        except Exception as e:
            print(f'⚠️  RP{rp}: {type(e).__name__}: {str(e)[:100]}')
            for p in tile_paths:
                if p.exists(): p.unlink()
            continue
    
    if not flood_maps_list:
        print(f'\n' + '='*80)
        print('💡 NEXT STEPS:')
        print('='*80)
        print('Your AOI is too large for the available system RAM.')
        print('To proceed, you need to:')
        print('')
        print('1. OPTION A: Reduce AOI extent')
        print('   - Edit Cell 1 to set a smaller AOI (just focus basin)')
        print('   - Re-run from Cell 1')
        print('')
        print('2. OPTION B: Increase available RAM')
        print('   - Use a machine with 16GB+ RAM')
        print('   - Or enable virtual memory/swap')
        print('')
        print('3. OPTION C: Sample/downsample the JRC tiles')
        print('   - Post-process to reduce resolution')
        print('   - (More complex, requires custom scripting)')
        print('='*80)
        raise RuntimeError('AOI too large for available RAM. See instructions above.')
    
    flood_maps = xr.concat(flood_maps_list, dim='return_period')
    flood_maps.to_netcdf(flood_maps_nc)
    print(f'\n✅ Saved flood_maps: {flood_maps_nc}')
    print(f'   Shape: {flood_maps.shape}')
    print(f'   Return periods: {list(flood_maps.return_period.values)}')

print(f'\n🌊 Done')


Selected JRC tiles: 1 | example: ['ID231_N20_E120']

⚠️  WARNING: AOI is large
   Bounds: (120.84583333333336, 15.86668124728736, 122.28750000000002, 18.33750000000002)
   Size: 1.44° × 2.47°
   Est. output: 2966×1730 (~0.0 GB)

✅ Using cached flood_maps: (7, 11999, 11999)
   Return periods: [10, 20, 50, 75, 100, 200, 500]

🌊 Done


In [10]:
# ── Depth helpers (verbatim from NB03 Cell 21) ───────────────────────────────
def read_raster(path):
    with rasterio.open(path) as src: return src.read(1), src.meta.copy()

def _coord_bounds(da, coord):
    v = da[coord].values; return float(np.nanmin(v)), float(np.nanmax(v))

def _sel_lon_lat(target, source):
    """Select a lon/lat slice of 'target' covering the full footprint of 'source' cells.

    The source grid uses cell CENTERS as coordinates.  Without correction the slice
    would stop at the outermost cell center, cutting off the outer half-cell on every
    edge (~0.025° for GloFAS v4).  We expand lo/hi by half the source resolution so
    the full cell footprint — center ± res/2 — is included in the JRC subset.
    """
    bounds = {}
    for coord in ['longitude', 'latitude']:
        lo, hi = _coord_bounds(source, coord)
        # Half-cell expansion: include the full footprint of boundary cells
        vals = source[coord].values
        res = float(np.median(np.abs(np.diff(vals)))) if len(vals) > 1 else 0.05
        lo -= res / 2
        hi += res / 2
        tv = target[coord].values; desc = tv[0] > tv[-1]
        bounds[coord] = slice(hi, lo) if desc else slice(lo, hi)
    return target.sel(bounds)

def _load_rp_da(nc):
    ds = xr.open_dataset(nc); da = ds['return_period']
    if 'latitude'  not in da.coords: da = da.assign_coords(latitude=ds['latitude'])
    if 'longitude' not in da.coords: da = da.assign_coords(longitude=ds['longitude'])
    return da

def _ensure_lon_lat(da):
    if 'lon' not in da.coords and 'longitude' in da.coords:
        da = da.assign_coords(lon=da['longitude'])
    if 'lat' not in da.coords and 'latitude' in da.coords:
        da = da.assign_coords(lat=da['latitude'])
    return da

def _match_coord_order(source, target):
    for coord in ['latitude','longitude']:
        sd = source[coord].values[0] > source[coord].values[-1]
        td = target[coord].values[0] > target[coord].values[-1]
        if sd != td: source = source.sortby(coord, ascending=not td)
    return source

def _sanitize_rp(da): return da.where(da >= 1.0)

def _fmt_bounds(da):
    la, lb = _coord_bounds(da,'latitude'); lo, lp = _coord_bounds(da,'longitude')
    return f'lat[{la:.4f},{lb:.4f}] lon[{lo:.4f},{lp:.4f}]'

def _apply_flopros(rp_rg, flopros_root):
    try:
        shp = flopros_root / 'FLOPROS_shp_V1/FLOPROS_shp_V1.shp'
        if not shp.exists(): download_flopros_database(str(flopros_root))
        gdf  = gpd.read_file(shp).to_crs('EPSG:4326')
        ovlp = int(gdf.intersects(AOI_BOUNDARY).sum())
        plist = []
        for i, ep in enumerate(rp_rg.event.values):
            ev_rp = _sanitize_rp(rp_rg.isel(event=i).copy(deep=True))
            ev_pr = petals_apply_flopros(gdf, ev_rp, layer='MerL_Riv') if ovlp > 0 else ev_rp
            plist.append(ev_pr.expand_dims({'event':[ep]}))
        return xr.concat(plist, dim='event')
    except Exception as e:
        print('⚠ FLOPROS failed:', e); return rp_rg

def _add_rp1_null(fm):
    """Add RP=1 null map if missing (on already-subsetted flood map to avoid MemoryError)."""
    if 1 not in fm['return_period'].values:
        null = xr.full_like(fm.isel(return_period=0), np.nan).expand_dims({'return_period': [1]})
        fm = xr.concat([null, fm], dim='return_period').sortby('return_period')
    return fm

# ── Depth TIF computation ─────────────────────────────────────────────────────
DEPTH_OUT_DIR = OUT_DIR / 'model/depth'
DEPTH_OUT_DIR.mkdir(parents=True, exist_ok=True)

def _compute_depth_tif(rp_da, ev_id, rp_label, apply_flopros):
    rp3 = _sanitize_rp(_ensure_lon_lat(rp_da.expand_dims({'event':[ev_id]})))
    # Subset flood_maps to event bbox; _sel_lon_lat expands by half a GloFAS cell
    # on each side so the full cell footprint is included (fixes half-cell clipping).
    fm  = _ensure_lon_lat(_sel_lon_lat(flood_maps, rp3))
    if fm.sizes.get('latitude',0)==0 or fm.sizes.get('longitude',0)==0:
        raise ValueError(f'No overlap RP/JRC. rp={_fmt_bounds(rp3)}')
    fm = fm.load()
    fm = fm.where(fm > -9000)   # mask JRC nodata sentinel (-9999) → NaN
    fm = _add_rp1_null(fm)
    rp3 = _match_coord_order(rp3, fm)
    # Regrid sparse GloFAS RP grid onto the dense JRC grid.
    # petals_regrid clips output to rp3's coordinate extent (bilinear can't extrapolate),
    # so the outer half-cell fringe added by _sel_lon_lat is absent from rp_rg_raw.
    # Fix: reindex onto fm's full grid first, then propagate boundary RP into the fringe.
    # Only pixels outside rp3's coordinate range (the fringe) are filled; interior NaN
    # (non-flooded gaps between active cells) is preserved unchanged.
    rp_rg_raw = petals_regrid(rp3, fm, method=REGRID_METHOD)
    rp_rg_raw = rp_rg_raw.reindex(latitude=fm.latitude, longitude=fm.longitude, method=None)
    lat_lo = float(rp3.latitude.min());  lat_hi = float(rp3.latitude.max())
    lon_lo = float(rp3.longitude.min()); lon_hi = float(rp3.longitude.max())
    rp_fringe_fill = (rp_rg_raw
                      .ffill('latitude').bfill('latitude')
                      .ffill('longitude').bfill('longitude'))
    in_domain = (
        (rp_rg_raw.latitude  >= lat_lo) & (rp_rg_raw.latitude  <= lat_hi) &
        (rp_rg_raw.longitude >= lon_lo) & (rp_rg_raw.longitude <= lon_hi))
    rp_rg = _sanitize_rp(rp_rg_raw.where(in_domain, rp_fringe_fill))
    if apply_flopros: rp_rg = _apply_flopros(rp_rg, OUT_DIR/'jrc')
    depth = petals_flood_depth(rp_rg, fm).isel(event=0)
    scen  = 'FLOPROS' if apply_flopros else 'NoProt'
    out   = DEPTH_OUT_DIR / f'depth_event__{ev_id}__{rp_label}__{scen}.tif'
    lats  = depth.latitude.values; lons = depth.longitude.values
    res_lon = float(np.mean(np.diff(lons))); res_lat = float(np.mean(np.diff(lats)))
    transform = rasterio.transform.from_origin(
        float(lons.min()-res_lon/2), float(lats.max()+abs(res_lat)/2), res_lon, abs(res_lat))
    meta = {'driver':'GTiff','height':depth.shape[0],'width':depth.shape[1],
            'count':1,'dtype':'float32','crs':'EPSG:4326','transform':transform,'nodata':np.nan}
    with rasterio.open(out, 'w', **meta) as dst:
        dst.write(depth.values.astype('float32'), 1)
    return out

print(f'⏳ Computing depth maps for {len(events_df)} events …')
depth_products = []
for _, row in events_df.iterrows():
    ev_id = row['event_id']
    for rp_label, nc in [('envelope', Path(row['envelope'])), ('peakday', Path(row['peakday']))]:
        rp_da = _load_rp_da(nc)
        for do_fl in ([False] + ([True] if ENABLE_FLOPROS_SCENARIOS else [])):
            out  = _compute_depth_tif(rp_da, ev_id, rp_label, do_fl)
            scen = 'FLOPROS' if do_fl else 'NoProt'
            depth_products.append({'event_id':ev_id,'rp_label':rp_label,'scenario':scen,'depth_tif':str(out)})
            print(f'  ✓ {ev_id} / {rp_label} / {scen}')

depth_df = pd.DataFrame(depth_products)
display(depth_df)

⏳ Computing depth maps for 4 events …
  ✓ EV_NEM2025 / envelope / NoProt
  ✓ EV_NEM2025 / envelope / FLOPROS
  ✓ EV_NEM2025 / peakday / NoProt
  ✓ EV_NEM2025 / peakday / FLOPROS
  ✓ EV_ULYSSES2020 / envelope / NoProt
  ✓ EV_ULYSSES2020 / envelope / FLOPROS
  ✓ EV_ULYSSES2020 / peakday / NoProt
  ✓ EV_ULYSSES2020 / peakday / FLOPROS
  ✓ EV_UWAN2025 / envelope / NoProt
  ✓ EV_UWAN2025 / envelope / FLOPROS
  ✓ EV_UWAN2025 / peakday / NoProt
  ✓ EV_UWAN2025 / peakday / FLOPROS
  ✓ EV_MARCE2024 / envelope / NoProt
  ✓ EV_MARCE2024 / envelope / FLOPROS
  ✓ EV_MARCE2024 / peakday / NoProt
  ✓ EV_MARCE2024 / peakday / FLOPROS


,event_id,rp_label,scenario,depth_tif
0,EV_NEM2025,envelope,NoProt,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...
1,EV_NEM2025,envelope,FLOPROS,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...
2,EV_NEM2025,peakday,NoProt,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...
3,EV_NEM2025,peakday,FLOPROS,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...
4,EV_ULYSSES2020,envelope,NoProt,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...
5,EV_ULYSSES2020,envelope,FLOPROS,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...
6,EV_ULYSSES2020,peakday,NoProt,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...
7,EV_ULYSSES2020,peakday,FLOPROS,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...
8,EV_UWAN2025,envelope,NoProt,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...
9,EV_UWAN2025,envelope,FLOPROS,C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL...


In [11]:
# ── Affected population at DEPTH_THRESHOLD_M ──────────────────────────────────
# Simplified vs NB03: one fixed threshold, no loop. Adds depth-band breakdown.

POP_OUT_DIR = OUT_DIR / 'population'
POP_OUT_DIR.mkdir(parents=True, exist_ok=True)

if not WORLDPOP_RASTER.exists():
    raise FileNotFoundError(f'WorldPop not found: {WORLDPOP_RASTER}')

_ref = depth_df[(depth_df['rp_label']=='envelope') & (depth_df['scenario']=='NoProt')].iloc[0]
depth0_arr, depth0_meta = read_raster(Path(_ref['depth_tif']))

with rasterio.open(WORLDPOP_RASTER) as src:
    pop_arr = src.read(1).astype('float32')
    pop_meta = src.meta.copy(); pop_nod = src.nodata

pop_reproj = np.full(depth0_arr.shape, np.nan, dtype='float32')
reproject(source=pop_arr, destination=pop_reproj,
          src_transform=pop_meta['transform'], src_crs=pop_meta.get('crs','EPSG:4326'),
          dst_transform=depth0_meta['transform'], dst_crs=depth0_meta.get('crs','EPSG:4326'),
          resampling=Resampling.bilinear, src_nodata=pop_nod, dst_nodata=np.nan)
print(f'WorldPop reprojected: {pop_reproj.shape}, total = {np.nansum(pop_reproj):,.0f}')

adm3_aoi = adm3_gdf[adm3_gdf.intersects(AOI_BOUNDARY)].copy()
print(f'ADM3 in AOI: {len(adm3_aoi)} municipalities')

def _pmask(poly, meta):
    return geometry_mask([poly], out_shape=(meta['height'],meta['width']),
                         transform=meta['transform'], invert=True)

poly_masks = {str(r['adm3_id']): _pmask(r.geometry, depth0_meta)
              for _, r in adm3_aoi.iterrows()}

# Build NB3-style admin raster for fast/consistent aggregation.
# Use stable internal integer codes so non-numeric ADM3 IDs are fully supported.
admin_id_raster = np.zeros(depth0_arr.shape, dtype='int32')
admin_code_to_id = {}
admin_code_to_name = {}
admin_ids_present = []
for code, (_, adm) in enumerate(adm3_aoi.iterrows(), start=1):
    mid_str = str(adm['adm3_id'])
    msk = poly_masks[mid_str]
    admin_id_raster[msk] = code
    admin_code_to_id[code] = mid_str
    admin_code_to_name[code] = adm['adm3_name']
    admin_ids_present.append(code)
admin_ids_present = np.array(sorted(set(admin_ids_present)), dtype='int32')

# Sanity check
try:
    with rasterio.open(WORLDPOP_RASTER) as src:
        out_img, _ = rio_mask(src, [AOI_BOUNDARY], crop=True, filled=False)
        a = out_img[0].astype('float32')
        if pop_nod is not None: a = np.where(a==pop_nod, np.nan, a)
        total_native = float(np.nansum(a))
    aoi_msk  = _pmask(AOI_BOUNDARY, depth0_meta)
    total_rp = float(np.nansum(np.where(aoi_msk, pop_reproj, np.nan)))
    if total_native > 0:
        ratio = total_rp / total_native
        flag  = '⚠️ >10% diff' if abs(ratio-1) > 0.10 else '✅ OK'
        print(f'Pop sanity: native={total_native:,.0f}  reproj={total_rp:,.0f}  ratio={ratio:.3f}  {flag}')
except Exception as e:
    print('⚠️ Pop sanity check skipped:', e)


def aggregate_affected_population_nb3_fast(
    pop_grid: np.ndarray,
    depth_grid: np.ndarray,
    admin_id_raster: np.ndarray,
    admin_ids_present: np.ndarray,
    thresholds_m: list[float],
    id_to_name: dict[int, str],
) -> pd.DataFrame:
    rows = []

    # NB3 parity: nodata contributes 0 and population is non-negative.
    pop_grid = np.asarray(pop_grid, dtype='float32')
    pop_grid = np.nan_to_num(pop_grid, nan=0.0)
    pop_grid = np.clip(pop_grid, 0.0, None)

    depth_grid = np.asarray(depth_grid, dtype='float32')
    admin_id_raster = np.asarray(admin_id_raster, dtype='int32')

    ids_flat = admin_id_raster.ravel()
    valid = ids_flat > 0
    ids_valid = ids_flat[valid]

    if ids_valid.size == 0:
        return pd.DataFrame(columns=['adm3_id', 'adm3_name', 'depth_thr_m', 'affected_pop'])

    for thr in thresholds_m:
        flooded = np.isfinite(depth_grid) & (depth_grid >= float(thr))
        flooded_pop = np.where(flooded, pop_grid, 0.0).ravel()
        flooded_pop_valid = flooded_pop[valid]

        sums = np.bincount(ids_valid, weights=flooded_pop_valid, minlength=int(admin_id_raster.max()) + 1)

        for adm_id in admin_ids_present:
            adm_id = int(adm_id)
            rows.append({
                'adm3_id': adm_id,
                'adm3_name': id_to_name.get(adm_id, str(adm_id)),
                'depth_thr_m': float(thr),
                'affected_pop': float(sums[adm_id]) if adm_id < len(sums) else 0.0,
            })

    return pd.DataFrame(rows, columns=['adm3_id', 'adm3_name', 'depth_thr_m', 'affected_pop'])


pop_adm3_rows = []
pop_summary_rows = []

for _, ev_row in events_df.iterrows():
    ev_id = ev_row['event_id']
    label = ev_row['label']
    d_row = depth_df[(depth_df['event_id']==ev_id) &
                     (depth_df['rp_label']=='envelope') &
                     (depth_df['scenario']=='NoProt')].iloc[0]
    darr, _ = read_raster(Path(d_row['depth_tif']))
    darr = darr.astype('float32')

    # Use NB3-style aggregation for threshold totals and admin-level sums.
    thr_main = float(DEPTH_THRESHOLD_M)
    thr_shallow = float(DEPTH_BAND_SHALLOW)
    thr_deep = float(DEPTH_BAND_MODERATE)
    thr_list = [thr_main, thr_shallow, thr_deep]

    df_imp = aggregate_affected_population_nb3_fast(
        pop_grid=pop_reproj,
        depth_grid=darr,
        admin_id_raster=admin_id_raster,
        admin_ids_present=admin_ids_present,
        thresholds_m=thr_list,
        id_to_name=admin_code_to_name,
    )

    if df_imp.empty:
        pop_t = pop_s = pop_m = pop_d = 0.0
    else:
        tot = df_imp.groupby('depth_thr_m')['affected_pop'].sum()
        pop_t = float(tot.get(thr_main, 0.0))
        pop_ge_shallow = float(tot.get(thr_shallow, 0.0))
        pop_ge_deep = float(tot.get(thr_deep, 0.0))

        pop_s = max(pop_t - pop_ge_shallow, 0.0)
        pop_m = max(pop_ge_shallow - pop_ge_deep, 0.0)
        pop_d = max(pop_ge_deep, 0.0)

    flooded = np.isfinite(darr) & (darr >= thr_main)
    valid_d = darr[flooded]
    dp99  = float(np.percentile(valid_d, 99)) if len(valid_d)>0 else DEPTH_THRESHOLD_M+0.01

    pop_summary_rows.append({'event_id':ev_id,'label':label,'pop_total':pop_t,
                              'pop_shallow':pop_s,'pop_moderate':pop_m,
                              'pop_deep':pop_d,'depth_p99':dp99})
    print(f'  ✅ {ev_id}: {pop_t:,.0f} people  '
          f'(sh={pop_s:,.0f}  mod={pop_m:,.0f}  deep={pop_d:,.0f})  p99={dp99:.1f}m')

    df_thr = df_imp[df_imp['depth_thr_m'] == thr_main]
    for _, r in df_thr.iterrows():
        code = int(r['adm3_id'])
        pop_adm3_rows.append({
            'event_id': ev_id,
            'adm3_id': admin_code_to_id.get(code, str(code)),
            'adm3_name': r['adm3_name'],
            'affected_pop': float(r['affected_pop']),
        })

pop_adm3_df    = pd.DataFrame(pop_adm3_rows)
pop_summary_df = pd.DataFrame(pop_summary_rows)
pop_adm3_df.to_csv(POP_OUT_DIR/'affected_population_by_adm3.csv', index=False)
pop_summary_df.to_csv(POP_OUT_DIR/'affected_population_summary.csv', index=False)
print('\nSaved population CSVs to:', POP_OUT_DIR)
display(pop_summary_df.round({'pop_total':0,'pop_shallow':0,'pop_moderate':0,'pop_deep':0,'depth_p99':2}))

WorldPop reprojected: (3000, 1800), total = 4,777,324
ADM3 in AOI: 118 municipalities
  ✅ EV_NEM2025: 145,484 people  (sh=91,200  mod=53,383  deep=900)  p99=2.8m
  ✅ EV_ULYSSES2020: 276,569 people  (sh=68,430  mod=52,654  deep=155,485)  p99=11.8m
  ✅ EV_UWAN2025: 243,699 people  (sh=139,207  mod=100,249  deep=4,243)  p99=2.5m
  ✅ EV_MARCE2024: 187,771 people  (sh=132,584  mod=49,888  deep=5,299)  p99=2.1m

Saved population CSVs to: C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\processed\event_viewer\Cagayan_01\2026-01-19_calib-test\population


,event_id,label,pop_total,pop_shallow,pop_moderate,pop_deep,depth_p99
0,EV_NEM2025,Shearline / NE Monsoon — Dec 2025,145484.0,91200.0,53383.0,900.0,2.75
1,EV_ULYSSES2020,TY Ulysses — Nov 2020,276569.0,68430.0,52654.0,155485.0,11.77
2,EV_UWAN2025,STY Uwan — Nov 2025,243699.0,139207.0,100249.0,4243.0,2.46
3,EV_MARCE2024,TY Marce — Nov 2024,187771.0,132584.0,49888.0,5299.0,2.11


In [12]:
# ── Population → Return Period via NB5 Watershed OEP Curve ────────────────────
# The OEP curve from the 10,000-year YLT simulation (NB5) is the authoritative
# frequency model.  Given an event's total affected population, we invert the
# watershed-level OEP curve to find the corresponding return period.
#
# Previous approach used the EVT2 GPD formula directly; this is now replaced
# because the OEP curve already incorporates the full simulation (EVT2 + spatial
# correlation + multi-event years), giving a more operationally consistent RP.
#
# Execution order: NB5 must run first to produce watershed_oep_curve.json.

import json as _json

# ── Load the OEP curve (hard-fail if missing) ────────────────────────────────
if not WATERSHED_OEP_CURVE_PATH.exists():
    raise FileNotFoundError(
        f"Watershed OEP curve not found: {WATERSHED_OEP_CURVE_PATH}\n"
        f"Run NB5 (Risk Profiles) before NB6 to generate this file."
    )

_oep_raw = _json.loads(WATERSHED_OEP_CURVE_PATH.read_text(encoding="utf-8"))
_oep_rp  = np.array(_oep_raw["rp"], dtype=float)       # RP values (e.g. 1,2,5,...,500)
_oep_pop = np.array(_oep_raw["oep_people"], dtype=float) # OEP people at each RP

print(f"✅ Watershed OEP curve loaded from NB5: {len(_oep_rp)} return periods")
print(f"   RP range: {_oep_rp[0]:.0f} – {_oep_rp[-1]:.0f}")
print(f"   OEP range: {_oep_pop[0]:,.0f} – {_oep_pop[-1]:,.0f} people")

# ── Inversion function: population → return period ───────────────────────────
def _pop_to_rp_oep(pop):
    """Invert the OEP curve: given affected population, return the RP.
    
    Uses log-RP interpolation on the OEP curve for smoother behaviour
    across the wide RP range (1–500 years).
    
    Returns:
        RP in years.  Values below the RP1 OEP return < 1.
        Values above the RP500 OEP return > 500 (extrapolated).
        Non-finite or non-positive inputs return NaN.
    """
    if not np.isfinite(pop) or pop <= 0:
        return float("nan")
    
    # OEP curve must be monotonically non-decreasing for interpolation
    # (enforce just in case of minor simulation noise)
    oep_mono = np.maximum.accumulate(_oep_pop)
    
    # Log-RP interpolation (more stable across wide range)
    log_rp = np.log(_oep_rp)
    rp = float(np.exp(np.interp(pop, oep_mono, log_rp)))
    
    return max(rp, 0.1)  # floor at 0.1 to avoid nonsensical near-zero values


# ── Apply to all named events ────────────────────────────────────────────────
pop_rp_rows = []
for _, row in pop_summary_df.iterrows():
    rp = _pop_to_rp_oep(row["pop_total"])
    pop_rp_rows.append({
        "event_id":  row["event_id"],
        "label":     row["label"],
        "pop_total": row["pop_total"],
        "rp_pop":    rp,
    })
    _rp_str = f"{rp:.1f}" if np.isfinite(rp) else "N/A"
    print(f"  {row['label']}")
    print(f"    People at risk : {row['pop_total']:>12,.0f}")
    print(f"    Return period  : {_rp_str:>12} years  (from NB5 OEP curve)")

pop_rp_df = pd.DataFrame(pop_rp_rows)
pop_rp_df.to_csv(POP_OUT_DIR / "population_return_period.csv", index=False)
print()
print("RP method: NB5 watershed OEP curve inversion (log-RP interpolation)")
display(pop_rp_df)


✅ Watershed OEP curve loaded from NB5: 11 return periods
   RP range: 1 – 500
   OEP range: 0 – 469,547 people
  Shearline / NE Monsoon — Dec 2025
    People at risk :      145,484
    Return period  :          1.7 years  (from NB5 OEP curve)
  TY Ulysses — Nov 2020
    People at risk :      276,569
    Return period  :         10.4 years  (from NB5 OEP curve)
  STY Uwan — Nov 2025
    People at risk :      243,699
    Return period  :          4.1 years  (from NB5 OEP curve)
  TY Marce — Nov 2024
    People at risk :      187,771
    Return period  :          2.0 years  (from NB5 OEP curve)

RP method: NB5 watershed OEP curve inversion (log-RP interpolation)


,event_id,label,pop_total,rp_pop
0,EV_NEM2025,Shearline / NE Monsoon — Dec 2025,145483.634334,1.715742
1,EV_ULYSSES2020,TY Ulysses — Nov 2020,276569.486719,10.369078
2,EV_UWAN2025,STY Uwan — Nov 2025,243699.233495,4.146572
3,EV_MARCE2024,TY Marce — Nov 2024,187771.069145,2.025118


In [13]:
# ── Save events_for_risk_matrix.json for NB5 Risk Matrix integration ─────────
# NB5 reads this file to plot named events as diamond markers on the EP chart.
# Must run NB6 before NB5 to generate this file; NB5 works without it (graceful fallback).

import json as _json
import numpy as np

_out = []
for _, _r in pop_rp_df.iterrows():
    _out.append({
        "event_id":  _r["event_id"],
        "label":     _r["label"],
        "pop_total": float(_r["pop_total"]) if np.isfinite(float(_r["pop_total"])) else None,
        "rp_pop":    float(_r["rp_pop"])    if np.isfinite(float(_r["rp_pop"]))    else None,
    })

EVENTS_FOR_RISKMATRIX_PATH.parent.mkdir(parents=True, exist_ok=True)
EVENTS_FOR_RISKMATRIX_PATH.write_text(_json.dumps(_out, indent=2), encoding="utf-8")
print(f"\u2705 events_for_risk_matrix.json written to: {EVENTS_FOR_RISKMATRIX_PATH}")
print(f"   {len(_out)} events:")
for _e in _out:
    _rp_str = f"{_e['rp_pop']:.1f} yr" if _e['rp_pop'] else "N/A"
    _pop_str = f"{_e['pop_total']:,.0f}" if _e['pop_total'] else "N/A"
    print(f"   {_e['label']}: pop={_pop_str}, RP={_rp_str}")


✅ events_for_risk_matrix.json written to: C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\processed\event_viewer\events_for_risk_matrix.json
   4 events:
   Shearline / NE Monsoon — Dec 2025: pop=145,484, RP=1.7 yr
   TY Ulysses — Nov 2020: pop=276,569, RP=10.4 yr
   STY Uwan — Nov 2025: pop=243,699, RP=4.1 yr
   TY Marce — Nov 2024: pop=187,771, RP=2.0 yr


In [14]:
from io import BytesIO
from PIL import Image

def _norm_id(x):
    try: return str(int(float(x)))
    except: return str(x)

def _render_depth_png(depth, vmin, vmax):
    # YlOrRd PNG transparent below vmin. D3: vmax = p99, no fixed cap.
    arr  = depth.astype('float32')
    mask = np.isfinite(arr) & (arr >= vmin)
    if vmax <= vmin: vmax = vmin + 0.01
    norm = (np.clip(arr, vmin, vmax) - vmin) / (vmax - vmin)
    rgba = (plt.get_cmap('YlOrRd')(np.nan_to_num(norm, nan=0.0)) * 255).astype(np.uint8)
    rgba[..., 3] = np.where(mask, 230, 0).astype(np.uint8)
    buf = BytesIO(); Image.fromarray(rgba, 'RGBA').save(buf, 'PNG')
    return base64.b64encode(buf.getvalue()).decode()

def _legend_bar_b64(w=220, h=12):
    grad  = np.linspace(0, 1, w)
    rgba  = (plt.get_cmap('YlOrRd')(grad) * 255).astype(np.uint8)
    rgba2 = np.tile(rgba[None,:,:], (h,1,1))
    buf   = BytesIO(); Image.fromarray(rgba2,'RGBA').save(buf,'PNG')
    return base64.b64encode(buf.getvalue()).decode()

def _bounds_wsen(meta):
    t = meta['transform']
    w = t.c; n = t.f; e = w+t.a*meta['width']; s = n+t.e*meta['height']
    return [min(w,e), min(s,n), max(w,e), max(s,n)]

def _severity(rp):
    """Map population return period to alert vocabulary (aligned with NB5 Risk Profile)."""
    if rp is None or not np.isfinite(rp): return 'sev-unknown', 'Severity unknown'
    if rp < 2:   return 'sev-moderate', 'Moderate event (RP1 — ≥100%/yr)'
    if rp < 5:   return 'sev-high',     'High event (RP2 — ≥50%/yr)'
    return 'sev-veryhigh', 'Very High event (RP5+)'

print('Collecting dashboard data ...')
dash_data = {}; all_pop_max = 0

for ev_id, ev in event_results.items():
    d_row = depth_df[(depth_df['event_id']==ev_id) &
                     (depth_df['rp_label']=='envelope') &
                     (depth_df['scenario']=='NoProt')].iloc[0]
    darr, dmeta = read_raster(Path(d_row['depth_tif'])); darr = darr.astype('float32')
    prow = pop_summary_df[pop_summary_df['event_id']==ev_id].iloc[0]
    rrow = pop_rp_df[pop_rp_df['event_id']==ev_id].iloc[0]
    vmin = DEPTH_THRESHOLD_M
    vmax = float(prow['depth_p99']) if np.isfinite(prow['depth_p99']) else vmin+1.0
    if vmax <= vmin: vmax = vmin + 0.01
    dpng   = _render_depth_png(darr, vmin, vmax)
    bounds = _bounds_wsen(dmeta)
    adm3_ev   = pop_adm3_df[pop_adm3_df['event_id']==ev_id]
    adm3_data = {_norm_id(r['adm3_id']): float(r['affected_pop']) for _, r in adm3_ev.iterrows()}
    all_pop_max = max(all_pop_max, max(adm3_data.values(), default=0))
    top10 = adm3_ev.sort_values('affected_pop', ascending=False).head(10)
    top_munis = [{'name':r['adm3_name'],'id':_norm_id(r['adm3_id']),'pop':float(r['affected_pop'])}
                 for _, r in top10.iterrows() if r['affected_pop'] > 0]
    timeline = [{'day':r['day'].strftime('%Y-%m-%d'),
                 'n_active':int(r['n_active']),
                 'pct':round(float(r['pct_active']),1),
                 'med_rp':round(float(r['median_rp']),1)}
                for _, r in ev['daily_stats'].iterrows()]
    rp_pop = float(rrow['rp_pop']) if np.isfinite(rrow['rp_pop']) else None
    sev_css, sev_label = _severity(rp_pop)
    annual_pct = round(100.0/rp_pop, 1) if rp_pop and rp_pop > 0 else None
    pk_str = ev['peak_day'].strftime('%d %b %Y')
    s_str  = ev['flood_days'][0].strftime('%d %b') if ev['flood_days'] else ev['date_start'].strftime('%d %b')
    e_str  = ev['flood_days'][-1].strftime('%d %b %Y') if ev['flood_days'] else ev['date_end'].strftime('%d %b %Y')
    dash_data[ev_id] = {
        'label':        ev['label'],
        'date_range':   f'{s_str} \u2013 {e_str}',
        'peak_day':     pk_str,
        'n_flood_days': len(ev['flood_days']),
        'depth_png':    dpng,
        'depth_bounds': bounds,
        'depth_vmin':   round(vmin, 2),
        'depth_vmax':   round(vmax, 2),
        'pop_total':    round(float(prow['pop_total'])),
        'pop_shallow':  round(float(prow['pop_shallow'])),
        'pop_moderate': round(float(prow['pop_moderate'])),
        'pop_deep':     round(float(prow['pop_deep'])),
        'rp_pop':       round(rp_pop, 1) if rp_pop else None,
        'annual_pct':   annual_pct,
        'sev_css':      sev_css,
        'sev_label':    sev_label,
        'adm3_data':    adm3_data,
        'top_munis':    top_munis,
        'n_munis':      int((adm3_ev['affected_pop'] > 100).sum()),
        'timeline':     timeline,
        'peak_day_iso': ev['peak_day'].strftime('%Y-%m-%d'),
    }
    print(f'  {ev_id}: p99={vmax:.1f}m  pop={prow["pop_total"]:,.0f}  rp={rp_pop}')

adm3_geojson = json.loads(adm3_aoi[['adm3_id','adm3_name','geometry']].to_json())
legend_b64   = _legend_bar_b64()
DD_JSON      = json.dumps(dash_data,   separators=(',',':'))
ADM3_JSON    = json.dumps(adm3_geojson, separators=(',',':'))
EVIDS_JSON   = json.dumps(list(dash_data.keys()), separators=(',',':'))
POP_MAX_VAL  = float(all_pop_max)
DEPTH_THR_M  = DEPTH_THRESHOLD_M

# ── HTML template (assembled as string to avoid triple-quote collision) ────────
_CSS = '''
:root{--navy:#0F2044;--red:#E31837;--blue:#2563EB;--amber:#D97706;--indigo:#4338CA}
*{box-sizing:border-box;margin:0;padding:0}
body{font-family:'IBM Plex Sans',system-ui,sans-serif;background:#F8FAFC;color:#1E293B;height:100vh;overflow:hidden}
.header{background:var(--navy);color:#fff;padding:0 20px;display:flex;align-items:center;gap:14px;height:52px;border-bottom:3px solid var(--red);flex-shrink:0}
.h-title{font-size:17px;font-weight:700;letter-spacing:-.01em}
.h-sub{font-size:11px;opacity:.65}
.layout{display:flex;height:calc(100vh - 52px)}
.sidebar{width:330px;min-width:310px;background:#fff;border-right:1px solid #E2E8F0;overflow-y:auto;padding:14px 13px;display:flex;flex-direction:column;gap:13px}
.sec-lbl{font-size:10px;font-weight:700;letter-spacing:.08em;text-transform:uppercase;color:#64748B;margin-bottom:4px}
select.ev-sel{width:100%;padding:8px 10px;border:1px solid #E2E8F0;border-radius:6px;font-size:13px;color:#0F172A;background:#fff;cursor:pointer}
select.ev-sel:focus{border-color:var(--blue);outline:none}
.date-strip{font-size:12px;color:#64748B;margin-top:3px}
.peak-strip{font-size:11px;color:#64748B;margin-top:1px}
.badge{padding:5px 12px;border-radius:16px;font-weight:700;font-size:12px;display:inline-block}
.sev-unknown{background:#E2E8F0;color:#475569}
.sev-moderate{background:#FEF9C3;color:#854D0E}
.sev-high{background:#FFEDD5;color:#9A3412}
.sev-veryhigh{background:#450a0a;color:#FCA5A5}
.sev-text{font-size:12px;color:#475569;line-height:1.6;margin-top:7px}
.kpi-row{display:flex;gap:8px}
.kpi-card{flex:1;border:1px solid #E2E8F0;border-radius:8px;overflow:hidden}
.kpi-stripe{height:4px}
.kpi-body{padding:9px 11px}
.kpi-lbl{font-size:10px;color:#64748B;font-weight:700;letter-spacing:.04em;text-transform:uppercase}
.kpi-val{font-size:22px;font-weight:800;color:#0F172A;margin-top:1px;line-height:1}
.kpi-sub{font-size:10px;color:#94A3B8;margin-top:2px}
.band-bar{width:100%;height:12px;border-radius:4px;overflow:hidden;display:flex;margin-top:5px}
.band-seg{height:100%}
.band-lbl{display:flex;justify-content:space-between;font-size:10px;color:#64748B;margin-top:3px}
.map-wrap{flex:1;position:relative}
#map{width:100%;height:100%}
.layer-toggle{position:absolute;top:10px;right:10px;z-index:1000;background:#fff;border-radius:7px;border:1px solid #E2E8F0;box-shadow:0 2px 8px rgba(0,0,0,.12);display:flex;overflow:hidden}
.lt-btn{padding:7px 14px;font-size:12px;font-weight:600;cursor:pointer;border:none;background:none;color:#64748B}
.lt-btn.active{background:var(--navy);color:#fff}
.depth-legend{position:absolute;bottom:28px;right:10px;z-index:1000;background:rgba(255,255,255,.93);padding:8px 10px;border-radius:7px;border:1px solid #E2E8F0;font-size:11px;color:#64748B}
.depth-legend img{display:block;margin:3px 0}
.muni-card{background:#F1F5F9;border-radius:7px;padding:10px 12px;font-size:12px;display:none}
.muni-card.visible{display:block}
.muni-name{font-weight:700;font-size:13px}
.muni-row{display:flex;justify-content:space-between;margin-top:5px}
.muni-val{font-weight:700;color:var(--blue)}
.summary-box{font-size:12px;line-height:1.65;color:#334155;background:#F1F5F9;border-radius:7px;padding:10px 12px}
.comp-table{width:100%;font-size:11px;border-collapse:collapse;margin-top:4px}
.comp-table td,.comp-table th{padding:3px 6px;text-align:left}
.comp-table th{color:#64748B;font-weight:600;border-bottom:1px solid #E2E8F0}
.comp-table tr:hover td{background:#F1F5F9;cursor:pointer}
.caveat{background:#FFFBEB;border:1px solid #FDE68A;border-radius:7px;padding:8px 11px;font-size:11px;color:#92400E;line-height:1.55}
'''

_JS = '''
const DD     = %%DD_JSON%%;
const ADM3   = %%ADM3_JSON%%;
const EV_IDS = %%EVIDS_JSON%%;
const POP_MAX = %%POP_MAX%%;
const LEG_B64 = 'data:image/png;base64,%%LEGEND%%';
const DEPTH_THR = %%DEPTH_THR%%;

const map = L.map('map',{zoomControl:true});
L.tileLayer('https://{s}.basemaps.cartocdn.com/light_all/{z}/{x}/{y}{r}.png',{
  attribution:'&copy; <a href="https://www.openstreetmap.org/copyright">OpenStreetMap</a> contributors &copy; <a href="https://carto.com/">CARTO</a>',maxZoom:19}).addTo(map);


let depthLayer=null, choroLayer=null, activeLayer='depth', selEvId=EV_IDS[0], selMuniId=null;

function popColour(pop){
  if(!pop||pop<=0) return '#F1F5F9';
  const t=Math.min(pop/POP_MAX,1.0);
  const r=Math.round(199-t*178),g=Math.round(210-t*193),b=Math.round(254-t*63);
  return `rgb(${r},${g},${b})`;
}

function _alertBadge(rp){
  if(!rp)return'';
  if(rp<2)return' <span class="badge sev-moderate" style="font-size:10px;padding:2px 6px">Moderate</span>';
  if(rp<5)return' <span class="badge sev-high" style="font-size:10px;padding:2px 6px">High</span>';
  return' <span class="badge sev-veryhigh" style="font-size:10px;padding:2px 6px">V.High</span>';
}

(function(){
  const sel=document.getElementById('evSel');
  EV_IDS.forEach(id=>{
    const opt=document.createElement('option');
    opt.value=id; opt.textContent=DD[id].label; sel.appendChild(opt);
  });
  sel.addEventListener('change',()=>{selEvId=sel.value;selMuniId=null;render();});
})();

function setLayer(l){
  activeLayer=l;
  document.getElementById('btnDepth').classList.toggle('active',l==='depth');
  document.getElementById('btnPop').classList.toggle('active',l==='pop');
  document.getElementById('depthLegend').style.display=l==='depth'?'block':'none';
  if(depthLayer) depthLayer.setOpacity(l==='depth'?1:0);
  if(choroLayer) choroLayer.setStyle(f=>_choroStyle(f));
}

function _choroStyle(feature){
  const id=String(feature.properties.adm3_id);
  const pop=(DD[selEvId]?.adm3_data||{})[id]||0;
  return{fillColor:popColour(pop),fillOpacity:activeLayer==='pop'?0.75:0.0,
         color:selMuniId===id?'#E31837':'#94A3B8',weight:selMuniId===id?2.5:0.6,opacity:0.8};
}

function _fmt(n){
  if(n>=1e6) return (n/1e6).toFixed(1)+'M';
  if(n>=1000) return Math.round(n/1000)+'K';
  return String(Math.round(n));
}

function render(){
  const ev=DD[selEvId]; if(!ev) return;

  document.getElementById('dateStrip').textContent=ev.date_range;
  document.getElementById('peakStrip').textContent=`Peak: ${ev.peak_day} \u2022 ${ev.n_flood_days} flood day(s) detected`;

  const badge=document.getElementById('sevBadge');
  badge.className=`badge ${ev.sev_css}`; badge.textContent=ev.sev_label;
  const st=document.getElementById('sevText');
  if(ev.rp_pop&&ev.annual_pct)
    st.textContent=`Estimated once every ${Math.round(ev.rp_pop)} years (${ev.annual_pct}% chance per year).`;
  else st.textContent='Return period unavailable (EVT2 data missing).';

  document.getElementById('kpiPop').textContent=_fmt(ev.pop_total);
  document.getElementById('kpiMunis').textContent=ev.n_munis;

  const tot=ev.pop_total||1;
  const pS=(100*ev.pop_shallow/tot).toFixed(1);
  const pM=(100*ev.pop_moderate/tot).toFixed(1);
  const pD=(100*ev.pop_deep/tot).toFixed(1);
  document.getElementById('bandBar').innerHTML=
    `<div class='band-seg' style='width:${pS}%;background:#FEF08A'></div>`+
    `<div class='band-seg' style='width:${pM}%;background:#FB923C'></div>`+
    `<div class='band-seg' style='width:${pD}%;background:#DC2626'></div>`;

  const mc=document.getElementById('muniCard');
  if(selMuniId){
    const pop=(ev.adm3_data||{})[selMuniId]||0;
    const feat=ADM3.features.find(f=>String(f.properties.adm3_id)===selMuniId);
    const name=feat?.properties?.adm3_name||selMuniId;
    document.getElementById('muniName').textContent=name;
    document.getElementById('muniPop').textContent=Math.round(pop).toLocaleString();
    mc.classList.add('visible');
  } else mc.classList.remove('visible');

  document.getElementById('legBar').src=LEG_B64;
  document.getElementById('legMin').textContent=ev.depth_vmin.toFixed(2)+' m';
  document.getElementById('legMax').textContent='\u2265 '+ev.depth_vmax.toFixed(1)+' m';

  if(depthLayer) map.removeLayer(depthLayer);
  const b=ev.depth_bounds;
  depthLayer=L.imageOverlay(
    'data:image/png;base64,'+ev.depth_png,
    [[b[1],b[0]],[b[3],b[2]]],
    {opacity:activeLayer==='depth'?1:0,interactive:false}).addTo(map);

  if(choroLayer) map.removeLayer(choroLayer);
  choroLayer=L.geoJSON(ADM3,{
    style:_choroStyle,
    onEachFeature:(feature,layer)=>{
      const id=String(feature.properties.adm3_id);
      const pop=(ev.adm3_data||{})[id]||0;
      const nm=feature.properties.adm3_name||id;
      layer.bindTooltip(`<b>${nm}</b><br>~${Math.round(pop).toLocaleString()} people at risk`,{sticky:true});
      layer.on('click',()=>{selMuniId=selMuniId===id?null:id;render();});
    },
  }).addTo(map);

  map.fitBounds([[b[1],b[0]],[b[3],b[2]]],{padding:[20,20]});

  const tm=(ev.top_munis||[]).filter(m=>m.pop>0);
  Plotly.newPlot('topMuniChart',[{
    type:'bar',orientation:'h',
    x:tm.map(m=>Math.round(m.pop)),y:tm.map(m=>m.name),
    marker:{color:'#2563EB',opacity:0.85},
    hovertemplate:'%{y}: %{x:,.0f}<extra></extra>',
  }],{
    margin:{t:2,b:20,l:120,r:10},height:180,
    xaxis:{tickformat:',d',gridcolor:'#F1F5F9'},
    yaxis:{autorange:'reversed',tickfont:{size:10}},
    paper_bgcolor:'transparent',plot_bgcolor:'transparent',showlegend:false,
  },{displayModeBar:false,responsive:true});

  const tl=ev.timeline||[];
  const pk=ev.peak_day_iso;
  const shapes=tl.some(d=>d.day===pk)?[{type:'line',x0:pk,x1:pk,y0:0,y1:1,xref:'x',yref:'paper',
    line:{color:'#D97706',width:1.5,dash:'dot'}}]:[];
  Plotly.newPlot('timelineChart',[{
    type:'bar',x:tl.map(d=>d.day),y:tl.map(d=>d.n_active),
    marker:{color:'#93C5FD'},name:'Active gauges',
    hovertemplate:'%{x}<br>%{y} active gauges<extra></extra>',
  }],{
    margin:{t:2,b:22,l:30,r:10},height:110,shapes,
    xaxis:{type:'date',tickformat:'%d %b',tickfont:{size:9}},
    yaxis:{tickfont:{size:9},gridcolor:'#F1F5F9'},
    paper_bgcolor:'transparent',plot_bgcolor:'transparent',showlegend:false,
    annotations:shapes.length?[{x:pk,y:1,xref:'x',yref:'paper',
      text:'Peak',showarrow:false,font:{size:9,color:'#D97706'},yanchor:'bottom'}]:[],
  },{displayModeBar:false,responsive:true});

  const popK=_fmt(ev.pop_total);
  const rpTxt=ev.rp_pop?`, estimated at a ${ev.rp_pop}-year return period`+
    ` (~${ev.annual_pct}% chance per year)`:'';
  document.getElementById('summaryBox').innerHTML=
    `During <b>${ev.label}</b>, the model estimates ~<b>${popK} people</b> at risk `+
    `(${ev.n_munis} municipalities). Peak: <b>${ev.peak_day}</b>${rpTxt}.`;

  const tbody=document.getElementById('compBody');
  tbody.innerHTML='';
  EV_IDS.filter(id=>id!==selEvId).forEach(id=>{
    const d=DD[id]; if(!d) return;
    const tr=document.createElement('tr');
    tr.innerHTML=`<td>${d.label}</td><td>${_fmt(d.pop_total)}</td><td>${d.rp_pop?d.rp_pop+'y':'N/A'}${_alertBadge(d.rp_pop)}</td>`;
    tr.onclick=()=>{document.getElementById('evSel').value=id;selEvId=id;selMuniId=null;render();};
    tbody.appendChild(tr);
  });
}

render();
'''

_HTML = (
    "<!DOCTYPE html>\n<html lang='en'>\n<head>\n"
    "<meta charset='UTF-8'>\n"
    "<meta name='viewport' content='width=device-width,initial-scale=1'>\n"
    "<title>" + BASIN_DISPLAY_NAME + " \u2014 Flood Event Viewer</title>\n"
    "<link rel='stylesheet' href='https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.css'/>\n"
    "<script src='https://cdnjs.cloudflare.com/ajax/libs/leaflet/1.9.4/leaflet.min.js'></script>\n"
    "<script src='https://cdn.plot.ly/plotly-2.20.0.min.js'></script>\n"
    "<style>" + _CSS + "</style>\n"
    "</head>\n<body>\n"
    "<div class='header'>"
    "<div><div class='h-title'>" + BASIN_DISPLAY_NAME + " \u2014 Flood Event Viewer</div>"
    "<div class='h-sub'>Modelled flood impact \u2022 Non-technical overview</div></div></div>\n"
    "<div class='layout'>\n"
    "<div class='sidebar'>\n"
    "  <div>\n"
    "    <div class='sec-lbl'>Flood Event</div>\n"
    "    <select class='ev-sel' id='evSel'></select>\n"
    "    <div class='date-strip' id='dateStrip'></div>\n"
    "    <div class='peak-strip' id='peakStrip'></div>\n"
    "  </div>\n"
    "  <div id='sevBlock'>\n"
    "    <div class='sec-lbl'>Event Severity</div>\n"
    "    <span class='badge' id='sevBadge'></span>\n"
    "    <div class='sev-text' id='sevText'></div>\n"
    "  </div>\n"
    "  <div>\n"
    "    <div class='sec-lbl'>Estimated People at Risk</div>\n"
    "    <div class='kpi-row'>\n"
    "      <div class='kpi-card'><div class='kpi-stripe' style='background:#2563EB'></div>"
    "        <div class='kpi-body'><div class='kpi-lbl'>People at risk</div>"
    "        <div class='kpi-val' id='kpiPop'>\u2014</div>"
    "        <div class='kpi-sub'>Depth \u2265 0.2m (forecast)</div></div></div>\n"
    "      <div class='kpi-card'><div class='kpi-stripe' style='background:#0D9488'></div>"
    "        <div class='kpi-body'><div class='kpi-lbl'>Municipalities</div>"
    "        <div class='kpi-val' id='kpiMunis'>\u2014</div>"
    "        <div class='kpi-sub'>&gt;100 people at risk</div></div></div>\n"
    "    </div>\n"
    "  </div>\n"
    "  <div>\n"
    "    <div class='sec-lbl'>Flood Depth Breakdown</div>\n"
    "    <div class='band-bar' id='bandBar'></div>\n"
    "    <div class='band-lbl'><span>&#127841; Shallow (&lt;0.5m)</span>"
    "      <span>&#127840; Moderate</span><span>&#128308; Deep (&gt;1.5m)</span></div>\n"
    "  </div>\n"
    "  <div class='muni-card' id='muniCard'>\n"
    "    <div class='muni-name' id='muniName'></div>\n"
    "    <div class='muni-row'><span>People at risk</span><span class='muni-val' id='muniPop'></span></div>\n"
    "  </div>\n"
    "  <div><div class='sec-lbl'>Most Affected Municipalities</div>"
    "    <div id='topMuniChart' style='height:180px'></div></div>\n"
    "  <div><div class='sec-lbl'>Discharge Activity by Day</div>"
    "    <div id='timelineChart' style='height:110px'></div></div>\n"
    "  <div><div class='sec-lbl'>Summary</div>"
    "    <div class='summary-box' id='summaryBox'></div></div>\n"
    "  <div><div class='sec-lbl'>How does this compare?</div>\n"
    "    <table class='comp-table' id='compTable'>\n"
    "      <thead><tr><th>Event</th><th>People at risk</th><th>RP / Alert</th></tr></thead>\n"
    "      <tbody id='compBody'></tbody></table></div>\n"
    "  <div class='caveat'>\u26a0\ufe0f Flood depths and population figures are model estimates. "
    "Actual impacts depend on local conditions not captured in the model.</div>\n"
    "</div>\n"   # /sidebar
    "<div class='map-wrap'>\n"
    "  <div id='map'></div>\n"
    "  <div class='layer-toggle'>\n"
    "    <button class='lt-btn active' id='btnDepth' onclick=\"setLayer('depth')\">Flood Depth</button>\n"
    "    <button class='lt-btn' id='btnPop' onclick=\"setLayer('pop')\">Affected Population</button>\n"
    "  </div>\n"
    "  <div class='depth-legend' id='depthLegend'>\n"
    "    <span id='legMin'></span><img id='legBar' width='220' height='12'><span id='legMax'></span>\n"
    "    <div style='margin-top:2px;font-size:10px'>Flood depth (m) \u2014 areas \u2265 0.2m</div>\n"
    "  </div>\n"
    "</div>\n"  # /map-wrap
    "</div>\n"  # /layout
    "<script>\n"
)

_js_final = (_JS
    .replace('%%DD_JSON%%',    DD_JSON)
    .replace('%%ADM3_JSON%%',  ADM3_JSON)
    .replace('%%EVIDS_JSON%%', EVIDS_JSON)
    .replace('%%POP_MAX%%',    str(POP_MAX_VAL))
    .replace('%%LEGEND%%',     legend_b64)
    .replace('%%DEPTH_THR%%',  str(DEPTH_THR_M))
    .replace('%%BASIN_NAME%%', repr(BASIN_DISPLAY_NAME))
)

html = _HTML + _js_final + '</script>\n</body>\n</html>'
DASHBOARD_FILENAME.write_text(html, encoding='utf-8')
print(f'✅ Dashboard written: {DASHBOARD_FILENAME}')
print(f'   Size: {DASHBOARD_FILENAME.stat().st_size / 1024:.0f} KB')
print('   Open in any browser — no server required.')


  EV_NEM2025: p99=2.8m  pop=145,484  rp=1.7157419418300446
  EV_ULYSSES2020: p99=11.8m  pop=276,569  rp=10.369077989769043
  EV_UWAN2025: p99=2.5m  pop=243,699  rp=4.146571766936889
  EV_MARCE2024: p99=2.1m  pop=187,771  rp=2.0251180474893804
✅ Dashboard written: C:\pipelines\GLOFAS_ImpactFloodForecasting_PHL\data\processed\event_viewer\Cagayan_01\2026-01-19_calib-test\event_viewer_dashboard.html
   Size: 8826 KB
   Open in any browser — no server required.
